In [2]:
from dotenv import load_dotenv

from app.services.fuel_price_service import FuelPriceService

load_dotenv()
from app.services.db_service import DbService
sql_db = DbService()
await sql_db.init_pool()

In [3]:
from app.handlers.navigation_handler import NavigationHandler
from app.services.internal_api.map_builder_api import MapBuilderApi
from app.services.template.telegram_template_service import TemplateService

template_service = TemplateService(sql_db, NavigationHandler(sql_db), MapBuilderApi("https://api.thebunkering.com", "https://api.thebunkering.com"))

port_fuel_price_service = FuelPriceService(sql_db)

In [19]:
from matplotlib import pyplot as plt
import pandas as pd
from app.data.dto.main.MabuxPortFuelPrice import MabuxPortFuelPriceDB
from app.data import emoji
from app.data.dto.messenger.ResponsePayload import MediaFile
from app.data.dto.main.QuoteRequestDB import QuoteRequestDB

import base64
from datetime import datetime
from io import BytesIO
from typing import Tuple, Optional, List
from jinja2 import Environment, FileSystemLoader
from weasyprint import HTML
from jinja2 import Environment, FileSystemLoader, select_autoescape

env = Environment(
    loader=FileSystemLoader("templates"),
    autoescape=select_autoescape(["html", "xml"])
)

def html_to_pdf(html: str) -> bytes:
    buffer = BytesIO()
    HTML(string=html).write_pdf(buffer)
    return buffer.getvalue()

async def render_supplier_request(
            user,
            quote_request: QuoteRequestDB,
            prices: List[MabuxPortFuelPriceDB]
    ) -> Tuple[str, bytes, object, str, List[str], Optional[bytes]]:

        # Get port information if port_id exists
        port = None
        if quote_request.port_id:
            port, _ = await sql_db.get_port_by_id(quote_request.port_id)

        # Map image
        indexed = []
        if port:
            indexed.append(port.to_indexed2(emoji.ANCHOR, "green", "medium", True))

        image_data, image_err = await template_service.map_image_api.render_map([], indexed)
        map_link = "https://www.britannica.com/science/world-map"  # map_image_api.get_route_map_link(str(quote_request.id))

        if image_data and not image_err:
            # Check if it's already base64 string
            if isinstance(image_data, str) and image_data.startswith('data:image'):
                # Already base64
                image_data = image_data.split(',')[1] if ',' in image_data else image_data
            elif isinstance(image_data, bytes):
                # Convert PNG bytes to base64
                image_data = base64.b64encode(image_data).decode('utf-8')
            else:
                print(f"Unexpected image data type: {type(image_data)}")

        prices_timeseries = {}
        if prices:
            for price in prices:
                prices_timeseries.setdefault(price.fuelName, []).append(price)

        # --- Build dataframe safely ---
        df = pd.DataFrame()
        if prices_timeseries:
            try:
                df_list = []
                for fuel, recs in prices_timeseries.items():
                    if recs:
                        df_list.append(pd.DataFrame([{"date": getattr(r, "date", None), fuel: getattr(r, "value", None)} for r in recs]))
                if df_list:
                    # Use merge instead of concat+groupby to get proper wide format
                    df = df_list[0].set_index('date')
                    for temp_df in df_list[1:]:
                        temp_df = temp_df.set_index('date')
                        df = df.merge(temp_df, left_index=True, right_index=True, how='outer')
                    df = df.sort_index()
                    df = df.groupby(level=0).last()
                    df = df.ffill()
            except Exception as e:
                #lines.append(f"\n⚠️ Failed to build dataframe: {str(e)}")
                pass

        prices_image_bytes = None
        if not df.empty:
            # Store the original index labels for x-axis
            original_index_labels = [i.strftime("%b, %d %Y") for i in df.index]

            # Reset index to numeric but keep it for plotting
            df_numeric = df.reset_index(drop=True)  # This creates 0,1,2,... index

            try:
                fig, ax = plt.subplots(figsize=(14, 7))

                # Plot using numeric index (0, 1, 2, ...)
                for fuel in df_numeric.columns:
                    ax.plot(df_numeric.index, df_numeric[fuel], marker="o", linewidth=2, label=fuel)

                    # Add data labels
                    for x, y in zip(df_numeric.index, df_numeric[fuel]):
                        if pd.notna(y):
                            ax.text(x, y, f"{y:.0f}", fontsize=12, ha="center", va="bottom")

                ax.set_ylabel("Price, $/mton")
                ax.set_title("Fuel Cost Timeseries, $/mton")
                ax.grid(True, linestyle="--", alpha=0.6)
                ax.legend()

                # Set x-axis ticks and labels
                ax.set_xticks(df_numeric.index)  # Set numeric positions
                ax.set_xticklabels(original_index_labels)  # Set original date labels

                # Rotate x-axis labels for better readability
                plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

                plt.tight_layout()

                buf = BytesIO()
                fig.savefig(buf, format='png', dpi=300)
                buf.seek(0)
                prices_image_bytes = buf.getvalue()
                buf.close()
                plt.close(fig)
            except Exception as e:
                #lines.append(f"\n⚠️ Failed to generate plot: {str(e)}")
                pass

        if prices_image_bytes:
            prices_image_bytes = base64.b64encode(prices_image_bytes).decode('utf-8')

        # Format ETA if exists
        eta_from_formatted = None
        if quote_request.eta_from:
            eta_from_formatted = quote_request.eta_from.strftime("%b %d, %Y")

        eta_to_formatted = None
        if quote_request.eta_from:
            eta_to_formatted = quote_request.eta_to.strftime("%b %d, %Y")

        # Prepare status for display and CSS class
        status_map = {
            "pending": {"display": "Pending", "class": "pending"},
            "requested": {"display": "Quote Requested", "class": "requested"},
            "quoted": {"display": "Quotes Received", "class": "completed"},
            "completed": {"display": "Completed", "class": "completed"},
            "cancelled": {"display": "Cancelled", "class": "cancelled"},
        }

        status_info = status_map.get(
            quote_request.status.lower(),
            {"display": quote_request.status, "class": "pending"}
        )

        # Prepare fuels data with specifications
        fuels_data = []
        total_cost = 0

        for f in quote_request.fuels:
            qty = f.quantity or 0
            price = f.price or 0
            cost = qty * price

            total_cost += cost

            fuels_data.append({
                "fuel_name": f.fuel_name,
                "quantity": qty,
                "price": price,
                "cost": cost
            })

        # Create email subject
        subject_parts = []
        if user and (user.first_name or user.last_name):
            subject_parts.append(f"{user.first_name or ''} {user.last_name or ''}".strip())
        if user and user.telegram_user_name:
            subject_parts.append(f"@{user.telegram_user_name}")
        if quote_request.vessel_name:
            subject_parts.append(f"- {quote_request.vessel_name}")

        base_subject = " ".join(subject_parts) if subject_parts else "Bunkering Bot"
        subject = f"⚓ FUEL QUOTE REQUEST: {base_subject}"

        # Get the Jinja2 template
        template = env.get_template("supplier_request.html")  # Make sure this template exists

        # Render the template
        html_content = template.render(
            request_id=quote_request.id,
            generation_date=datetime.now().strftime("%d %B %Y"),
            total_cost=total_cost,
            user=user,
            vessel_name=quote_request.vessel_name,
            vessel_imo=quote_request.vessel_imo,
            eta_from=eta_from_formatted,
            eta_to=eta_to_formatted,
            port=port,
            fuels=fuels_data,
            remark=quote_request.remark,
            image=image_data,
            map_link=map_link,
            prices_image_bytes=prices_image_bytes
        )

        # Generate PDF
        html_content_bytes = html_to_pdf(html_content)

        # Create filename
        filename_parts = []
        if port:
            filename_parts.append(port.port_name.replace(" ", "_"))
        if quote_request.vessel_name:
            filename_parts.append(quote_request.vessel_name.replace(" ", "_"))
        filename_parts.append("Quote_Request")
        if quote_request.eta_from:
            filename_parts.append(quote_request.eta_from.strftime("%Y%m%d"))

        if quote_request.eta_to:
            filename_parts.append("")
            filename_parts.append(quote_request.eta_to.strftime("%Y%m%d"))


        filename = "_".join(filename_parts) if filename_parts else "Quote_Request"

        # Create MediaFile object
        # Adjust import as needed
        file_obj = MediaFile(
            filename=filename,
            content=html_content_bytes,
        )

        # Return compatibility values (empty images list, no image data)
        return html_content, html_content_bytes, file_obj, subject, [], None

In [5]:
import uuid
user,user_err = await sql_db.get_user_by_telegram_name("semyon_spb")
session, err = await sql_db.get_or_create_session(user.id)
quote_r, err = await sql_db.get_quote_request_by_id(str(session.route_id))
fuels, err = await sql_db.get_available_fuels()
port, err = await sql_db.get_port_by_id(quote_r.port_id)
prices = await port_fuel_price_service.get_port_fuel_prices(
    port,
    fuels,
    datetime.now().date(),
)


In [7]:
len(prices)

127

In [ ]:
datetime.now().date()

In [ ]:
port

In [20]:
html_content, html_content_bytes, file_obj, subject, coords, err = await render_supplier_request(user, quote_r, prices)
pdf_bytes = html_to_pdf(html_content)
with open("file.pdf", 'wb+') as fp:
    fp.write(pdf_bytes)

In [9]:
prices_timeseries = {}
if prices:
    for price in prices:
        prices_timeseries.setdefault(price.fuelName, []).append(price)

# --- Build dataframe safely ---
df = pd.DataFrame()
if prices_timeseries:
    try:
        df_list = []
        for fuel, recs in prices_timeseries.items():
            if recs:
                df_list.append(pd.DataFrame([{"date": getattr(r, "date", None), fuel: getattr(r, "value", None)} for r in recs]))
        if df_list:
            # Use merge instead of concat+groupby to get proper wide format
            df = df_list[0].set_index('date')
            for temp_df in df_list[1:]:
                temp_df = temp_df.set_index('date')
                df = df.merge(temp_df, left_index=True, right_index=True, how='outer')
            df = df.sort_index()
            df = df.groupby(level=0).last()
            df = df.ffill()
    except Exception as e:
        #lines.append(f"\n⚠️ Failed to build dataframe: {str(e)}")
        pass

prices_image_bytes = None
if not df.empty:
    # Store the original index labels for x-axis
    original_index_labels = [i.strftime("%b, %d %Y") for i in df.index]

    # Reset index to numeric but keep it for plotting
    df_numeric = df.reset_index(drop=True)  # This creates 0,1,2,... index

    try:
        fig, ax = plt.subplots(figsize=(14, 7))

        # Plot using numeric index (0, 1, 2, ...)
        for fuel in df_numeric.columns:
            ax.plot(df_numeric.index, df_numeric[fuel], marker="o", linewidth=2, label=fuel)

            # Add data labels
            for x, y in zip(df_numeric.index, df_numeric[fuel]):
                if pd.notna(y):
                    ax.text(x, y, f"{y:.0f}", fontsize=12, ha="center", va="bottom")

        ax.set_ylabel("Price, $/mton")
        ax.set_title("Fuel Cost Timeseries, $/mton")
        ax.grid(True, linestyle="--", alpha=0.6)
        ax.legend()

        # Set x-axis ticks and labels
        ax.set_xticks(df_numeric.index)  # Set numeric positions
        ax.set_xticklabels(original_index_labels)  # Set original date labels

        # Rotate x-axis labels for better readability
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

        plt.tight_layout()

        buf = BytesIO()
        fig.savefig(buf, format='png', dpi=300)
        buf.seek(0)
        prices_image_bytes = buf.getvalue()
        buf.close()
        plt.close(fig)
    except Exception as e:
        #lines.append(f"\n⚠️ Failed to generate plot: {str(e)}")
        pass

In [11]:
with open("p.png", 'wb+') as fp:
    fp.write(prices_image_bytes)

In [17]:
base64_image = base64.b64encode(prices_image_bytes).decode('utf-8')
base64_image

'...'